# Yambda 50M likes. CDN-Adaptive

## Imports and config

In [ ]:
import json
import random
import itertools
import gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    "data_dir": "../data/",
    "interactions_file": "likes.parquet",
    "artist_file": "artist_item_mapping.parquet",
    "album_file": "album_item_mapping.parquet",

    "max_users": None,
    "min_user_interactions": 3,
    "min_item_interactions": 5,

    "embed_dim": 64,
    "artist_emb_dim": 32,
    "album_emb_dim": 32,
    "tower_hidden": [256, 128, 64],
    "expert_hidden": 64,

    "batch_size": 4096,
    "grad_clip": 1.0,
    "temperature": 1.0,

    "final_epochs": 20,
    "cdn_tune_epochs": 3,
    "resample_head_fraction": 0.20,

    "tune_fast": True,
    "tune_val_fraction": 1.00,
    "skip_tune_if_cached": True,
    "cache_path": "best_params.json",

    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 8192,

    "head_fraction": 0.20,
    "head_fractions_sweep": [0.001, 0.005, 0.01, 0.05, 0.20],
    
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print(f"Final: {len(CFG['seeds'])} seed(s) × {CFG['final_epochs']} epochs")

device: cuda
Final: 5 seed(s) × 20 epochs


## 1. Load data

In [ ]:
def find_file(data_dir, name):
    data_dir = Path(data_dir)
    candidates = [
        data_dir / name,
        data_dir / f"{name}.parquet",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"Could not find {name}. Tried: {candidates}")


def normalize_interaction_columns(df: pl.DataFrame):
    rename = {}

    for c in df.columns:
        lc = c.lower()

        if lc in {"uid", "user_id", "userid", "user"}:
            rename[c] = "uid"
        elif lc in {"item_id", "itemid", "track_id", "trackid", "song_id"}:
            rename[c] = "item_id"
        elif lc in {"timestamp", "ts", "time", "event_timestamp", "datetime"}:
            rename[c] = "timestamp"

    return df.rename(rename)


def normalize_item_side_columns(df: pl.DataFrame):
    rename = {}

    for c in df.columns:
        lc = c.lower()

        if lc in {"item_id", "itemid", "track_id", "trackid"}:
            rename[c] = "item_id"
        elif lc in {"artist_id", "artistid"}:
            rename[c] = "artist_id"
        elif lc in {"album_id", "albumid"}:
            rename[c] = "album_id"

    return df.rename(rename)


DATA_DIR = Path(CFG["data_dir"])
INTERACTIONS_PATH = find_file(DATA_DIR, CFG["interactions_file"])
ARTIST_PATH = find_file(DATA_DIR, CFG["artist_file"])
ALBUM_PATH = find_file(DATA_DIR, CFG["album_file"])

print("interactions:", INTERACTIONS_PATH)
print("artists:", ARTIST_PATH)
print("albums:", ALBUM_PATH)

interactions = pl.read_parquet(INTERACTIONS_PATH)
interactions = normalize_interaction_columns(interactions)

print("raw interactions:", interactions.shape)
print("columns:", interactions.columns)
print(interactions.head())

required = {"uid", "item_id"}
missing = required - set(interactions.columns)
assert not missing, f"Missing required columns {missing}. Available: {interactions.columns}"

if "timestamp" not in interactions.columns:
    print("[WARN] timestamp not found: using row index as timestamp")
    interactions = interactions.with_row_index("timestamp")

interactions = interactions.select(["uid", "item_id", "timestamp"])

interactions = interactions.sort("timestamp").unique(subset=["uid", "item_id"], keep="first")

print("after dedup:", interactions.shape)

interactions: ../data/likes.parquet
artists: ../data/artist_item_mapping.parquet
albums: ../data/album_item_mapping.parquet
raw interactions: (881456, 4)
columns: ['uid', 'timestamp', 'item_id', 'is_organic']
shape: (5, 4)
┌─────┬───────────┬─────────┬────────────┐
│ uid ┆ timestamp ┆ item_id ┆ is_organic │
│ --- ┆ ---       ┆ ---     ┆ ---        │
│ u32 ┆ u32       ┆ u32     ┆ u8         │
╞═════╪═══════════╪═════════╪════════════╡
│ 100 ┆ 44755     ┆ 732449  ┆ 1          │
│ 100 ┆ 1155860   ┆ 6568592 ┆ 0          │
│ 100 ┆ 1259125   ┆ 5411243 ┆ 1          │
│ 100 ┆ 1260005   ┆ 7371186 ┆ 0          │
│ 100 ┆ 1263935   ┆ 4943655 ┆ 0          │
└─────┴───────────┴─────────┴────────────┘
after dedup: (844690, 3)


## 2. Filtering

In [ ]:
item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
interactions = interactions.join(good_items, on="item_id", how="semi")
print("after item-core:", interactions.shape)

user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
interactions = interactions.join(good_users, on="uid", how="semi")
print("after user-core:", interactions.shape)

if CFG["max_users"] is not None:
    users_sub = interactions.select("uid").unique().sample(n=int(CFG["max_users"]), seed=CFG["seed"])
    interactions = interactions.join(users_sub, on="uid", how="semi")
    print(f"after max_users={CFG['max_users']}:", interactions.shape)

    item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
    good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
    interactions = interactions.join(good_items, on="item_id", how="semi")

    user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
    good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
    interactions = interactions.join(good_users, on="uid", how="semi")

print("final filtered:", interactions.shape)
print("users:", interactions["uid"].n_unique())
print("items:", interactions["item_id"].n_unique())

after item-core: (621417, 3)
after user-core: (620105, 3)
final filtered: (620105, 3)
users: 7102
items: 33145


## 3. ID mapping and leave-one-out split

In [ ]:
user_mapping = interactions.select("uid").unique().sort("uid").with_row_index(name="u_idx")
item_mapping = interactions.select("item_id").unique().sort("item_id").with_row_index(name="i_idx")

inter = (
    interactions.join(user_mapping, on="uid", how="inner")
    .join(item_mapping, on="item_id", how="inner")
    .select(["u_idx", "i_idx", "timestamp"])
    .sort(["u_idx", "timestamp"])
)

NUM_USERS = user_mapping.height
NUM_ITEMS = item_mapping.height

inter = inter.with_columns(
    [
        pl.arange(0, pl.len()).over("u_idx").alias("pos"),
        pl.len().over("u_idx").alias("hist_len"),
    ]
)

train_df = inter.filter(pl.col("pos") < pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
val_df = inter.filter(pl.col("pos") == pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
test_df = inter.filter(pl.col("pos") == pl.col("hist_len") - 1).select(["u_idx", "i_idx"])

train_pairs = train_df.to_numpy().astype(np.int64)

val_np = val_df.to_numpy().astype(np.int64)
test_np = test_df.to_numpy().astype(np.int64)

val_u, val_i = val_np[:, 0], val_np[:, 1]
test_u, test_i = test_np[:, 0], test_np[:, 1]

print(f"NUM_USERS={NUM_USERS:,}")
print(f"NUM_ITEMS={NUM_ITEMS:,}")
print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")
print(f"catalog share @50 = {50 / NUM_ITEMS:.6f}")

assert len(train_pairs) > 0
assert len(val_u) == NUM_USERS
assert len(test_u) == NUM_USERS

NUM_USERS=7,102
NUM_ITEMS=33,145
train=605,901 val=7,102 test=7,102
catalog share @50 = 0.001509


## 4. Build item features: artist_id and album_id

In [ ]:
item_features_df = item_mapping.select(["item_id", "i_idx"])

artists = pl.read_parquet(ARTIST_PATH)
artists = normalize_item_side_columns(artists)

print("artists shape:", artists.shape)
print("artists columns:", artists.columns)

if "item_id" not in artists.columns or "artist_id" not in artists.columns:
    raise ValueError(f"Bad artist mapping columns: {artists.columns}")

artists_one = (
    artists.select(["item_id", "artist_id"])
    .drop_nulls(["item_id", "artist_id"])
    .group_by("item_id")
    .agg(pl.col("artist_id").first())
)

item_features_df = item_features_df.join(artists_one, on="item_id", how="left")

albums = pl.read_parquet(ALBUM_PATH)
albums = normalize_item_side_columns(albums)

print("albums shape:", albums.shape)
print("albums columns:", albums.columns)

if "item_id" not in albums.columns or "album_id" not in albums.columns:
    raise ValueError(f"Bad album mapping columns: {albums.columns}")

albums_one = (
    albums.select(["item_id", "album_id"])
    .drop_nulls(["item_id", "album_id"])
    .group_by("item_id")
    .agg(pl.col("album_id").first())
)

item_features_df = item_features_df.join(albums_one, on="item_id", how="left")

item_features_df = item_features_df.sort("i_idx")

unique_artists = (
    item_features_df.select("artist_id")
    .drop_nulls()
    .unique()
    .sort("artist_id")
    .with_row_index(name="artist_idx", offset=1)
)

item_features_df = item_features_df.join(unique_artists, on="artist_id", how="left").with_columns(
    pl.col("artist_idx").fill_null(0).cast(pl.Int64)
)

unique_albums = (
    item_features_df.select("album_id")
    .drop_nulls()
    .unique()
    .sort("album_id")
    .with_row_index(name="album_idx", offset=1)
)

item_features_df = item_features_df.join(unique_albums, on="album_id", how="left").with_columns(
    pl.col("album_idx").fill_null(0).cast(pl.Int64)
)

ITEM_ARTIST = item_features_df["artist_idx"].to_numpy().astype(np.int64)
ITEM_ALBUM = item_features_df["album_idx"].to_numpy().astype(np.int64)

NUM_ARTISTS = int(ITEM_ARTIST.max()) + 1
NUM_ALBUMS = int(ITEM_ALBUM.max()) + 1

print("NUM_ITEMS:", NUM_ITEMS)
print("NUM_ARTISTS:", NUM_ARTISTS)
print("NUM_ALBUMS:", NUM_ALBUMS)
print("artist unknown share:", float((ITEM_ARTIST == 0).mean()))
print("album unknown share:", float((ITEM_ALBUM == 0).mean()))

item_artist_t = torch.from_numpy(ITEM_ARTIST).long().to(device)
item_album_t = torch.from_numpy(ITEM_ALBUM).long().to(device)

artists shape: (9271906, 2)
artists columns: ['artist_id', 'item_id']
albums shape: (9651644, 2)
albums columns: ['album_id', 'item_id']
NUM_ITEMS: 33145
NUM_ARTISTS: 9321
NUM_ALBUMS: 23839
artist unknown share: 0.0
album unknown share: 3.017046311660884e-05


## 5. Known items. Head/tail split

In [ ]:
train_user_items = [set() for _ in range(NUM_USERS)]

for u, i in train_pairs:
    train_user_items[int(u)].add(int(i))

known_val = [set(s) for s in train_user_items]

known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)
order = np.argsort(-item_freq)

n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))

head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"nonzero train items={np.sum(item_freq > 0):,}")
print(f"zero train items={np.sum(item_freq == 0):,}")
print(f"val true head share={head_mask[val_i].mean():.4f}")
print(f"test true head share={head_mask[test_i].mean():.4f}")


def make_head_mask(item_freq: np.ndarray, head_fraction):
    num_items = len(item_freq)
    n_head = max(1, int(num_items * head_fraction))
    order = np.argsort(-item_freq)

    current_head_mask = np.zeros(num_items, dtype=bool)
    current_head_mask[order[:n_head]] = True

    return current_head_mask

head=6,629 tail=26,516
nonzero train items=33,145
zero train items=0
val true head share=0.5579
test true head share=0.5368


## 6. Rebalancing

In [ ]:
def make_regularizer_pairs(seed=0):
    rng = np.random.default_rng(seed)

    pairs_by_item = defaultdict(list)
    for u, i in train_pairs:
        pairs_by_item[int(i)].append((int(u), int(i)))

    local_head_mask = make_head_mask(item_freq, CFG["resample_head_fraction"])

    tail_freq = item_freq[~local_head_mask]
    tail_freq = tail_freq[tail_freq > 0]

    max_tail_freq = int(tail_freq.max()) if len(tail_freq) else 1

    reg_pairs = []

    for i in range(NUM_ITEMS):
        pairs = pairs_by_item.get(i, [])

        if not pairs:
            continue

        if local_head_mask[i] and len(pairs) > max_tail_freq:
            idx = rng.choice(len(pairs), size=max_tail_freq, replace=False)
            reg_pairs.extend([pairs[j] for j in idx])
        else:
            reg_pairs.extend(pairs)

    rng.shuffle(reg_pairs)

    return np.asarray(reg_pairs, dtype=np.int64), max_tail_freq


reg_pairs, max_tail_freq = make_regularizer_pairs(seed=CFG["seed"])

print(f"Omega_m train pairs={len(train_pairs):,}")
print(f"Omega_r regularizer pairs={len(reg_pairs):,}")
print(f"max_tail_freq={max_tail_freq:,}")
print(f"Omega_r ratio={len(reg_pairs) / max(len(train_pairs), 1):.3f}")

Omega_m train pairs=605,901
Omega_r regularizer pairs=389,341
max_tail_freq=22
Omega_r ratio=0.643


## 7. Dataset and CDN-Adaptive model

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


def make_loader(pairs: np.ndarray, batch_size, shuffle=True):
    return DataLoader(
        PairDataset(pairs),
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
    )


class MLP(nn.Module):
    def __init__(self, in_dim, hidden: list, dropout=0.0):
        super().__init__()

        layers = []

        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h

        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CDNAdaptive(nn.Module):
    def __init__(
        self,
        num_users,
        num_items,
        num_artists,
        num_albums,
        embed_dim=64,
        artist_emb_dim=32,
        album_emb_dim=32,
        hidden: list = [256, 128, 64],
        expert_hidden=64,
        dropout=0.0,
        n_mem_experts=1,
        n_gen_experts=1,
    ):
        super().__init__()

        self.out_dim = hidden[-1]

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.user_shared = MLP(embed_dim, [hidden[0]], dropout)
        self.user_main = MLP(hidden[0], hidden[1:], dropout)
        self.user_reg = MLP(hidden[0], hidden[1:], dropout)
        self.user_main_ln = nn.LayerNorm(self.out_dim)
        self.user_reg_ln = nn.LayerNorm(self.out_dim)

        self.item_emb = nn.Embedding(num_items, embed_dim)
        self.artist_emb = nn.Embedding(num_artists, artist_emb_dim)
        self.album_emb = nn.Embedding(num_albums, album_emb_dim)

        self.mem_experts = nn.ModuleList([MLP(embed_dim, [expert_hidden], dropout) for _ in range(n_mem_experts)])
        gen_in_dim = artist_emb_dim + album_emb_dim
        self.gen_experts = nn.ModuleList([MLP(gen_in_dim, [expert_hidden], dropout) for _ in range(n_gen_experts)])

        self.mem_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gen_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gate = nn.Linear(1, n_mem_experts + n_gen_experts)
        self.item_ln = nn.LayerNorm(self.out_dim)

        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_base(self, u: torch.Tensor):
        return self.user_shared(self.user_emb(u))

    def user_vec_main(self, u: torch.Tensor):
        return self.user_main_ln(self.user_main(self.user_base(u)))

    def user_vec_reg(self, u: torch.Tensor):
        return self.user_reg_ln(self.user_reg(self.user_base(u)))

    def item_vec_moe(self, i: torch.Tensor):
        id_x = self.item_emb(i)
        mem = torch.stack([self.mem_proj(expert(id_x)) for expert in self.mem_experts], dim=1)
        gen_x = torch.cat([self.artist_emb(item_artist_t[i]), self.album_emb(item_album_t[i])], dim=-1)
        gen = torch.stack([self.gen_proj(expert(gen_x)) for expert in self.gen_experts], dim=1)
        experts = torch.cat([mem, gen], dim=1)
        gate_in = item_pop_t[i].unsqueeze(-1)
        weights = torch.softmax(self.gate(gate_in), dim=-1).unsqueeze(-1)
        return self.item_ln((experts * weights).sum(dim=1))

    def user_vec(self, u: torch.Tensor):
        return self.user_vec_main(u)

    def item_vec(self, i: torch.Tensor):
        return self.item_vec_moe(i)

    def cdn_losses(
        self,
        main_users,
        main_items,
        reg_users,
        reg_items,
        epoch,
        total_epochs,
        gamma_0,
        gamma_1,
    ):
        xm = F.normalize(self.user_vec_main(main_users), dim=-1, eps=1e-6)
        ym = F.normalize(self.item_vec_moe(main_items), dim=-1, eps=1e-6)
        xr = F.normalize(self.user_vec_reg(reg_users), dim=-1, eps=1e-6)
        yr = F.normalize(self.item_vec_moe(reg_items), dim=-1, eps=1e-6)

        logits_m = torch.clamp((xm @ ym.T) / CFG.get("temperature", 1.0), -20.0, 20.0)
        logits_r = torch.clamp((xr @ yr.T) / CFG.get("temperature", 1.0), -20.0, 20.0)

        if not torch.isfinite(logits_m).all():
            raise RuntimeError("NaN/Inf in main logits")
        if not torch.isfinite(logits_r).all():
            raise RuntimeError("NaN/Inf in regularizer logits")

        labels_m = torch.arange(logits_m.size(0), device=logits_m.device)
        labels_r = torch.arange(logits_r.size(0), device=logits_r.device)

        gamma_m = (gamma_0 + gamma_1 * item_norm_pop_t[main_items]).clamp(min=1.001)
        gamma_r = (gamma_0 + gamma_1 * item_norm_pop_t[reg_items]).clamp(min=1.001)
        alpha_m = (1.0 - (epoch / (gamma_m * total_epochs)) ** 2).clamp(0.0, 1.0)
        alpha_r = (1.0 - (epoch / (gamma_r * total_epochs)) ** 2).clamp(0.0, 1.0)

        loss_m_per = F.cross_entropy(logits_m, labels_m, reduction="none")
        loss_r_per = F.cross_entropy(logits_r, labels_r, reduction="none")
        loss = (alpha_m * loss_m_per).mean() + ((1.0 - alpha_r) * loss_r_per).mean()
        return loss, loss_m_per.mean(), loss_r_per.mean()


item_pop = np.log1p(item_freq.astype(np.float32))
item_pop = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-6)
item_pop_t = torch.from_numpy(item_pop.astype(np.float32)).to(device)

_lp = np.log1p(item_freq.astype(np.float32))
item_norm_pop = (_lp - _lp.min()) / (_lp.max() - _lp.min() + 1e-6)
item_norm_pop_t = torch.from_numpy(item_norm_pop.astype(np.float32)).to(device)

main_loader = make_loader(train_pairs, CFG["batch_size"], shuffle=True)
reg_loader = make_loader(reg_pairs, CFG["batch_size"], shuffle=True)

print("CDN-Adaptive helpers ready")
print("main batches:", len(main_loader))
print("reg batches:", len(reg_loader))
print(f"item_norm_pop: min={item_norm_pop.min():.3f} max={item_norm_pop.max():.3f} mean={item_norm_pop.mean():.3f}")

CDN-Adaptive helpers ready
main batches: 147
reg batches: 95
item_norm_pop: min=0.000 max=1.000 mean=0.314


## 8. Evaluation

In [ ]:
@torch.no_grad()
def evaluate_full_corpus(
    model,
    users,
    true_items,
    known_user_items,
    head_mask,
    ks,
    item_freq,
    user_batch_size=128,
    item_batch_size=8192,
):
    model.eval()

    ranks_all, ranks_head, ranks_tail = [], [], []
    max_k = max(ks)
    recommended_by_k = {k: [] for k in ks}

    item_vectors = []

    for s in range(0, NUM_ITEMS, item_batch_size):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        v = F.normalize(model.item_vec(idx), dim=-1, eps=1e-6)

        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on item batch {s}:{e}")

        item_vectors.append(v.cpu())

    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    for start in range(0, len(users), user_batch_size):
        end = min(start + user_batch_size, len(users))
        bu = users[start:end]
        bi = true_items[start:end]
        ut = torch.tensor(bu, device=device, dtype=torch.long)
        uvec = F.normalize(model.user_vec(ut), dim=-1, eps=1e-6)
        scores = (uvec @ item_vectors.T).cpu().numpy()

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u, true_i = int(u), int(true_i)
            srow = scores[row].copy()

            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            true_score = srow[true_i]
            eps = 1e-12
            rank = int((srow > true_score + eps).sum()) + int((np.abs(srow - true_score) <= eps).sum()) - 1
            ranks_all.append(rank)
            (ranks_head if head_mask[true_i] else ranks_tail).append(rank)

            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]

            else:
                top_idx = np.argsort(-srow)

            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    def agg_accuracy(ranks):
        a = np.asarray(ranks, dtype=np.int64)
        out = {}

        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan

            else:
                hits = a < k
                out[f"HR@{k}"] = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(hits, 1.0 / np.log2(a + 2), 0.0).mean()

        return out

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())
    popularity = np.log1p(item_freq.astype(np.float64))
    long_tail_metrics = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)
        catalog_coverage = len(unique_recs) / NUM_ITEMS
        tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items if num_tail_items > 0 else np.nan
        avg_popularity = popularity[recs].mean()
        tail_ratio = tail_mask[recs].mean()
        n_users_eval = recs.shape[0]

        if n_users_eval <= 1:
            personalization = np.nan

        else:
            exposure_counts = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise_intersections = np.sum(exposure_counts * (exposure_counts - 1) / 2)
            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs
            personalization = 1.0 - avg_overlap / k

        long_tail_metrics[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail_metrics[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail_metrics[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail_metrics[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail_metrics[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail_metrics,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics):
    print("counts:", metrics.get("counts", {}))

    for split in ["overall", "head", "tail"]:
        print(f"[{split}]", metrics[split])

    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])


@torch.no_grad()
def evaluate_head_tail_sweep(model, method_name, seed, head_fractions, test_u, test_i, known_test, item_freq, ks):
    rows = []
    model.eval()
    for hf in head_fractions:
        print("\n" + "=" * 80)
        print(f"{method_name} | seed={seed} | head_fraction={hf} ({100 * hf:.3f}%)")
        print("=" * 80)

        current_head_mask = make_head_mask(item_freq=item_freq, head_fraction=hf)
        num_head_items = int(current_head_mask.sum())
        num_tail_items = int((~current_head_mask).sum())
        test_head_share = float(current_head_mask[test_i].mean())
        test_tail_share = float((~current_head_mask[test_i]).mean())

        print(f"num_head_items: {num_head_items:,}")
        print(f"num_tail_items: {num_tail_items:,}")
        print(f"test_head_share: {test_head_share:.4f}")
        print(f"test_tail_share: {test_tail_share:.4f}")
        metrics = evaluate_full_corpus(
            model=model,
            users=test_u,
            true_items=test_i,
            known_user_items=known_test,
            head_mask=current_head_mask,
            ks=ks,
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )
        print_metrics(metrics)

        row = {
            "method": method_name,
            "seed": seed,
            "head_fraction": hf,
            "head_percent": 100.0 * hf,
            "num_head_items": num_head_items,
            "num_tail_items": num_tail_items,
            "test_head_share": test_head_share,
            "test_tail_share": test_tail_share,
        }
        
        for split in ("overall", "head", "tail"):
            for key, value in metrics[split].items():
                row[f"{split}_{key}"] = value

        for key, value in metrics.get("long_tail", {}).items():
            row[key] = value

        for key, value in metrics.get("counts", {}).items():
            row[f"count_{key}"] = value

        rows.append(row)

    return rows

## 9. Training helpers

In [ ]:
def mean_alpha_for_epoch(epoch, total_epochs, gamma_0, gamma_1):
    gammas = (gamma_0 + gamma_1 * item_norm_pop_t).clamp(min=1.001)
    alphas = (1.0 - (epoch / (gammas * total_epochs)) ** 2).clamp(0.0, 1.0)
    return float(alphas.mean().item())


def train_one_cdn_epoch(model, optimizer, main_loader, reg_loader, epoch, total_epochs, gamma_0, gamma_1):
    model.train()
    total_loss, total_n = 0.0, 0
    reg_iter = iter(reg_loader)

    for main_users, main_items in main_loader:
        try:
            reg_users, reg_items = next(reg_iter)

        except StopIteration:
            reg_iter = iter(reg_loader)
            reg_users, reg_items = next(reg_iter)

        main_users = main_users.to(device, non_blocking=True)
        main_items = main_items.to(device, non_blocking=True)
        reg_users = reg_users.to(device, non_blocking=True)
        reg_items = reg_items.to(device, non_blocking=True)

        loss, loss_m, loss_r = model.cdn_losses(
            main_users,
            main_items,
            reg_users,
            reg_items,
            epoch,
            total_epochs,
            gamma_0,
            gamma_1,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite CDN loss at epoch={epoch}: {loss.item()}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()

        bs = main_users.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(total_n, 1)


def train_cdn(
    model,
    optimizer,
    epochs,
    gamma_0,
    gamma_1,
    main_loader,
    reg_loader,
    val_u,
    val_i,
    known_val,
    head_mask,
    item_freq,
    seed_tag="",
    track_val=True,
):
    best_val_hr50, best_state, history = -1.0, None, []
    k_track = CFG["eval_k"][-1]
    for ep in range(1, epochs + 1):
        avg_loss = train_one_cdn_epoch(
            model,
            optimizer,
            main_loader,
            reg_loader,
            ep,
            epochs,
            gamma_0,
            gamma_1,
            desc=f"CDN-Adaptive {seed_tag}",
        )
        log = {
            "epoch": ep,
            "mean_alpha": mean_alpha_for_epoch(ep, epochs, gamma_0, gamma_1),
            "train_loss": avg_loss,
        }

        if track_val:
            vm = evaluate_full_corpus(
                model=model,
                users=val_u,
                true_items=val_i,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )

            hr = vm["overall"].get(f"HR@{k_track}", -1.0)
            log["val_HR@50"] = hr

            if hr > best_val_hr50:
                best_val_hr50 = hr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            print(f"  CDN-Adaptive ep{ep} mean_alpha={log['mean_alpha']:.4f} loss={avg_loss:.4f} val HR@{k_track}={hr:.4f}")

        else:
            print(f"  CDN-Adaptive ep{ep} mean_alpha={log['mean_alpha']:.4f} loss={avg_loss:.4f}")

        history.append(log)

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_hr50, history

## 10. Grid search

In [ ]:
LR_GRID = [0.001]
DROPOUT_GRID = [0.1]
WD_GRID = [0.0]

GAMMA0_GRID = [1.5, 3.0, 10.0]
GAMMA1_GRID = [0.0, 1.0, 3.0, 7.0]
EXPERT_COUNT_GRID = [2]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID, GAMMA0_GRID, GAMMA1_GRID, EXPERT_COUNT_GRID))
k_main = CFG["eval_k"][-1]
cache_path = Path(CFG["cache_path"])

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")

else:
    frac = float(CFG.get("tune_val_fraction", 1.0))
    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[idx], val_i[idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["cdn_tune_epochs"]
    print(f"Tuning CDN-Adaptive {len(combos)} trials × {tune_ep} ep (val subset: {len(val_u_t):,}/{len(val_u):,})")
    best_hr, best_hp = -1.0, None
    for ti, (lr, dr, wd, gamma_0, gamma_1, n_experts) in enumerate(combos, 1):
        set_seed(CFG["seed"])

        m = CDNAdaptive(
            NUM_USERS,
            NUM_ITEMS,
            NUM_ARTISTS,
            NUM_ALBUMS,
            embed_dim=CFG["embed_dim"],
            artist_emb_dim=CFG["artist_emb_dim"],
            album_emb_dim=CFG["album_emb_dim"],
            hidden=CFG["tower_hidden"],
            expert_hidden=CFG["expert_hidden"],
            dropout=dr,
            n_mem_experts=n_experts,
            n_gen_experts=n_experts,
        ).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
        status, hr = "ok", -1.0

        try:
            _, hr, _ = train_cdn(
                model=m,
                optimizer=opt,
                epochs=tune_ep,
                gamma_0=gamma_0,
                gamma_1=gamma_1,
                main_loader=main_loader,
                reg_loader=reg_loader,
                val_u=val_u_t,
                val_i=val_i_t,
                known_val=known_val,
                head_mask=head_mask,
                item_freq=item_freq,
                seed_tag=f"trial{ti}",
                track_val=True,
            )
            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")

        except RuntimeError as e:
            status = "failed"
            print(f"  CDN-Adaptive trial {ti} FAILED: {e}")

        print(
            f"  CDN-Adaptive trial {ti:4d}/{len(combos)} lr={lr:.0e} dr={dr} wd={wd:.0e} gamma_0={gamma_0:g} gamma_1={gamma_1:g} experts={n_experts} -> val HR@{k_main}={hr:.2f}%"
        )

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {
                "lr": lr,
                "dropout": dr,
                "weight_decay": wd,
                "gamma_0": gamma_0,
                "gamma_1": gamma_1,
                "n_experts": n_experts,
                f"val_HR@{k_main}": hr,
            }

        del m, opt
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial.")

    cache_path.write_text(json.dumps(best_hp, indent=2))
    print(f"\nBest CDN-Adaptive val HR@{k_main}={best_hr:.2f}% -> {best_hp}")
    print(f"Saved best hparams: {cache_path}")

Loaded cached hparams: {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'gamma_0': 1.5, 'gamma_1': 0.0, 'n_experts': 2, 'val_HR@50': 1.591101098282174}


## 11. Final training

In [ ]:
all_rows, all_test, all_sweep_rows = [], [], []
print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"CDN-Adaptive seed {seed}")
    print("=" * 80)

    set_seed(seed)

    reg_pairs_seed, max_tail_freq_seed = make_regularizer_pairs(seed=seed)
    main_loader_seed = make_loader(train_pairs, CFG["batch_size"], shuffle=True)
    reg_loader_seed = make_loader(reg_pairs_seed, CFG["batch_size"], shuffle=True)
    print(f"Omega_r seed={seed}: {len(reg_pairs_seed):,} pairs, max_tail_freq={max_tail_freq_seed:,}")

    model = CDNAdaptive(
        NUM_USERS,
        NUM_ITEMS,
        NUM_ARTISTS,
        NUM_ALBUMS,
        embed_dim=CFG["embed_dim"],
        artist_emb_dim=CFG["artist_emb_dim"],
        album_emb_dim=CFG["album_emb_dim"],
        hidden=CFG["tower_hidden"],
        expert_hidden=CFG["expert_hidden"],
        dropout=best_hp["dropout"],
        n_mem_experts=int(best_hp["n_experts"]),
        n_gen_experts=int(best_hp["n_experts"]),
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_hp["lr"], weight_decay=best_hp["weight_decay"])

    best_state, best_val_hr50, _ = train_cdn(
        model=model,
        optimizer=optimizer,
        epochs=CFG["final_epochs"],
        gamma_0=float(best_hp["gamma_0"]),
        gamma_1=float(best_hp["gamma_1"]),
        main_loader=main_loader_seed,
        reg_loader=reg_loader_seed,
        val_u=val_u,
        val_i=val_i,
        known_val=known_val,
        head_mask=head_mask,
        item_freq=item_freq,
        seed_tag=f"seed{seed}",
        track_val=True,
    )
    print(f"seed {seed} best val HR@50: {best_val_hr50:.4f}")

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )

    print("TEST METRICS")
    print_metrics(test_metrics)
    all_test.append(test_metrics)

    row = {
        "method": "CDN-Adaptive",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "gamma_0": best_hp["gamma_0"],
        "gamma_1": best_hp["gamma_1"],
        "n_experts": best_hp["n_experts"],
        "final_epochs": CFG["final_epochs"],
        "best_val_HR@50": best_val_hr50,
    }
    
    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value

    for key, value in test_metrics.get("long_tail", {}).items():
        row[f"test_{key}"] = value

    for key, value in test_metrics.get("counts", {}).items():
        row[f"test_count_{key}"] = value

    all_rows.append(row)
    sweep_rows_seed = evaluate_head_tail_sweep(
        model=model,
        method_name="CDN-Adaptive",
        seed=seed,
        head_fractions=CFG["head_fractions_sweep"],
        test_u=test_u,
        test_i=test_i,
        known_test=known_test,
        item_freq=item_freq,
        ks=CFG["eval_k"],
    )
    all_sweep_rows.extend(sweep_rows_seed)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_rows)
sweep_df = pd.DataFrame(all_sweep_rows)
metrics_df

Using best_hp: {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'gamma_0': 1.5, 'gamma_1': 0.0, 'n_experts': 2, 'val_HR@50': 1.591101098282174}

CDN-Adaptive seed 0
Omega_r seed=0: 389,341 pairs, max_tail_freq=22
  CDN-Adaptive ep1 mean_alpha=0.9989 loss=8.2044 val HR@50=1.2954
  CDN-Adaptive ep2 mean_alpha=0.9956 loss=8.0039 val HR@50=1.1828
  CDN-Adaptive ep3 mean_alpha=0.9900 loss=7.9328 val HR@50=0.7885
  CDN-Adaptive ep4 mean_alpha=0.9822 loss=7.9139 val HR@50=1.2532
  CDN-Adaptive ep5 mean_alpha=0.9722 loss=7.9015 val HR@50=1.1687
  CDN-Adaptive ep6 mean_alpha=0.9600 loss=7.8889 val HR@50=1.4925
  CDN-Adaptive ep7 mean_alpha=0.9456 loss=7.8791 val HR@50=1.6615
  CDN-Adaptive ep8 mean_alpha=0.9289 loss=7.8704 val HR@50=1.6615
  CDN-Adaptive ep9 mean_alpha=0.9100 loss=7.8639 val HR@50=1.8164
  CDN-Adaptive ep10 mean_alpha=0.8889 loss=7.8586 val HR@50=1.8446
  CDN-Adaptive ep11 mean_alpha=0.8656 loss=7.8542 val HR@50=1.7882
  CDN-Adaptive ep12 mean_alpha=0.8400 loss=7.8500 val HR@

,method,seed,lr,dropout,weight_decay,gamma_0,gamma_1,n_experts,final_epochs,best_val_HR@50,...,test_TailRatio@10,test_Personalization@10,test_CatalogCoverage@50,test_TailCoverage@50,test_AvgPopularity@50,test_TailRatio@50,test_Personalization@50,test_count_overall,test_count_head,test_count_tail
0,CDN-Adaptive,0,0.001,0.1,0.0,1.5,0.0,2,20,2.154323,...,77.424669,99.947096,98.820335,98.891235,2.607461,77.478175,99.768324,7102,3812,3290
1,CDN-Adaptive,1,0.001,0.1,0.0,1.5,0.0,2,20,2.112081,...,78.014644,99.952451,98.156585,98.023835,2.584764,78.404393,99.796210,7102,3812,3290
2,CDN-Adaptive,2,0.001,0.1,0.0,1.5,0.0,2,20,2.098001,...,76.772740,99.950641,97.577312,97.220546,2.617824,77.123064,99.793404,7102,3812,3290
3,CDN-Adaptive,3,0.001,0.1,0.0,1.5,0.0,2,20,2.562658,...,77.616164,99.949273,98.192789,98.087947,2.596195,77.970431,99.781577,7102,3812,3290
4,CDN-Adaptive,4,0.001,0.1,0.0,1.5,0.0,2,20,2.238806,...,76.002534,99.945465,97.987630,97.895610,2.633845,76.338496,99.761918,7102,3812,3290


## 12. Final table

In [ ]:
def make_final_summary_table(metrics_df, sweep_df, method_name, save_path=None):
    if len(sweep_df) == 0:
        raise ValueError("sweep_df is empty")

    selected_metrics = [
        "overall_HR@50",
        "overall_NDCG@50",
        "head_HR@50",
        "head_NDCG@50",
        "tail_HR@50",
        "tail_NDCG@50",
        "CatalogCoverage@50",
        "TailCoverage@50",
        "AvgPopularity@50",
        "TailRatio@50",
        "Personalization@50",
    ]

    rows = []
    for head_fraction, group in sweep_df.groupby("head_fraction"):
        group = group.copy()
        row = {
            "method": method_name,
            "head_share": f"{100 * float(head_fraction):.3f}%",
            "head_fraction": float(head_fraction),
            "num_seeds": (group["seed"].nunique() if "seed" in group.columns else len(group)),
            "num_head_items": (int(group["num_head_items"].iloc[0]) if "num_head_items" in group.columns else np.nan),
            "num_tail_items": (int(group["num_tail_items"].iloc[0]) if "num_tail_items" in group.columns else np.nan),
        }
        if "test_head_share" in group.columns:
            row["test_head_share"] = f"{100 * float(group['test_head_share'].mean()):.2f}%"

        if "test_tail_share" in group.columns:
            row["test_tail_share"] = f"{100 * float(group['test_tail_share'].mean()):.2f}%"

        if metrics_df is not None and len(metrics_df) > 0:
            for hp_col in [
                "lr",
                "dropout",
                "weight_decay",
                "gamma_0",
                "gamma_1",
                "n_experts",
                "final_epochs",
            ]:
                if hp_col in metrics_df.columns:
                    vals = metrics_df[hp_col].dropna().unique()
                    row[hp_col] = vals[0] if len(vals) == 1 else ", ".join(map(str, vals))

            if "best_val_HR@50" in metrics_df.columns:
                vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
                if len(vals) > 0:
                    mean = float(np.mean(vals))
                    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
                    row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

        for metric in selected_metrics:
            if metric not in group.columns:
                continue

            vals = group[metric].dropna().to_numpy(dtype=float)
            if len(vals) == 0:
                row[metric] = "nan"
                continue

            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            row[metric] = f"{mean:.4f} ± {std:.4f}" if "AvgPopularity" in metric else f"{mean:.2f} ± {std:.2f}"

        rows.append(row)

    summary_df = pd.DataFrame(rows).sort_values("head_fraction").reset_index(drop=True)
    first_cols = [
        "method",
        "head_share",
        "head_fraction",
        "num_seeds",
        "num_head_items",
        "num_tail_items",
        "test_head_share",
        "test_tail_share",
        "lr",
        "dropout",
        "weight_decay",
        "gamma_0",
        "gamma_1",
        "n_experts",
        "final_epochs",
        "best_val_HR@50",
    ]
    ordered_cols = [c for c in first_cols + selected_metrics if c in summary_df.columns]
    other_cols = [c for c in summary_df.columns if c not in ordered_cols]
    summary_df = summary_df[ordered_cols + other_cols]

    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")

    return summary_df


summary_df = make_final_summary_table(
    metrics_df,
    sweep_df,
    method_name="CDN-Adaptive",
    save_path="summary.csv",
)
summary_df

saved: yambda_cdn_adaptive_summary.csv


,method,head_share,head_fraction,num_seeds,num_head_items,num_tail_items,test_head_share,test_tail_share,lr,dropout,...,overall_NDCG@50,head_HR@50,head_NDCG@50,tail_HR@50,tail_NDCG@50,CatalogCoverage@50,TailCoverage@50,AvgPopularity@50,TailRatio@50,Personalization@50
0,CDN-Adaptive,0.100%,0.001,5,33,33112,2.65%,97.35%,0.001,0.1,...,0.49 ± 0.05,1.06 ± 0.65,0.27 ± 0.15,1.92 ± 0.19,0.50 ± 0.05,98.15 ± 0.45,98.15 ± 0.45,2.6080 ± 0.0190,99.86 ± 0.01,99.78 ± 0.02
1,CDN-Adaptive,0.500%,0.005,5,165,32980,6.43%,93.57%,0.001,0.1,...,0.49 ± 0.05,1.40 ± 0.63,0.33 ± 0.14,1.93 ± 0.19,0.50 ± 0.05,98.15 ± 0.45,98.14 ± 0.45,2.6080 ± 0.0190,99.24 ± 0.05,99.78 ± 0.02
2,CDN-Adaptive,1.000%,0.010,5,331,32814,10.38%,89.62%,0.001,0.1,...,0.49 ± 0.05,1.19 ± 0.35,0.28 ± 0.08,1.98 ± 0.20,0.52 ± 0.05,98.15 ± 0.45,98.14 ± 0.45,2.6080 ± 0.0190,98.60 ± 0.08,99.78 ± 0.02
3,CDN-Adaptive,5.000%,0.050,5,1657,31488,28.56%,71.44%,0.001,0.1,...,0.49 ± 0.05,1.73 ± 0.32,0.43 ± 0.08,1.96 ± 0.18,0.52 ± 0.05,98.15 ± 0.45,98.12 ± 0.49,2.6080 ± 0.0190,93.84 ± 0.33,99.78 ± 0.02
4,CDN-Adaptive,20.000%,0.200,5,6629,26516,53.68%,46.32%,0.001,0.1,...,0.49 ± 0.05,1.64 ± 0.28,0.41 ± 0.06,2.19 ± 0.13,0.58 ± 0.04,98.15 ± 0.45,98.02 ± 0.60,2.6080 ± 0.0190,77.46 ± 0.79,99.78 ± 0.02
